# KeelNet Stage 4: Unsupported-Confidence Control Template

Use this notebook as the Stage 4 working file for the official team workflow:

1. edit locally in VS Code
2. open this notebook in browser Google Colab
3. rerun the setup cell after pushing code changes
4. save artifacts to Google Drive or your local runtime project folder

This notebook keeps the Stage 4 notes, implementation hints, run commands, and shareable outputs in one place.


## Stage Notes

### Goal

Reduce confident unsupported answers without collapsing usefulness.

### Scope

- input: answer, support signal, confidence signal
- output: better control over unsupported confident answering

### Main Change

- add an explicit penalty or control mechanism for confident unsupported outputs

### Main Metrics

- unsupported-answer rate
- supported-answer rate
- answer `F1`
- abstain `F1`

### Handoff Condition

Do not move to the next stage until unsupported confident answers go down without answer quality collapsing.


<h2 style="color: #1d4ed8;">1. Setup And Sync</h2>

Run this cell in either hosted Google Colab or Google Colab connected to a local Jupyter runtime.

What it does:

- hosted Colab: mounts Drive, loads `HF_TOKEN` from Colab Secrets if available, clones or updates `/content/KeelNet`, checks out the stage branch, and installs the project
- local runtime: reuses your local repo, uses a local project folder for artifacts, reads `HF_TOKEN` from the environment if available, and installs the project into the current kernel environment

Path reminder:

- hosted Colab defaults: repo `/content/KeelNet`, project folder `/content/drive/MyDrive/KeelNet`
- local runtime defaults: repo `/content/KeelNet` if present, otherwise your current local checkout; project folder `/content/KeelNet-local`
- optional overrides: `KEELNET_REPO_DIR` and `KEELNET_PROJECT_DIR`


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


GIT_REPO_URL = "https://github.com/naufalkmd/KeelNet.git"
GIT_BRANCH = 'stage/04-unsupported-confidence-control'
HOSTED_COLAB_PROJECT_DIR = Path("/content/drive/MyDrive/KeelNet")
DEFAULT_LOCAL_PROJECT_DIR = Path("/content/KeelNet-local")
DEFAULT_LOCAL_REPO_DIR = Path("/content/KeelNet")


def detect_runtime_mode() -> str:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return "local-runtime"
    return "hosted-colab"


RUNTIME_MODE = detect_runtime_mode()
IS_HOSTED_COLAB = RUNTIME_MODE == "hosted-colab"
PROJECT_STORAGE_LABEL = "Drive project dir" if IS_HOSTED_COLAB else "Local project dir"


def run_setup(cmd, *, cwd: Path | None = None) -> None:
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    subprocess.run(rendered, check=True, cwd=str(cwd) if cwd else None)


def configure_project_storage() -> Path:
    if IS_HOSTED_COLAB:
        from google.colab import drive

        project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(HOSTED_COLAB_PROJECT_DIR)))
        drive.mount("/content/drive", force_remount=False)
        if not str(project_dir).startswith("/content/drive/"):
            raise ValueError(
                f"KEELNET_PROJECT_DIR must point inside /content/drive in hosted Colab, got: {project_dir}"
            )
        project_dir.mkdir(parents=True, exist_ok=True)
        print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
        return project_dir

    project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(DEFAULT_LOCAL_PROJECT_DIR))).expanduser().resolve()
    project_dir.mkdir(parents=True, exist_ok=True)
    print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
    return project_dir


def configure_hf_token() -> None:
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN already set in the environment.")
        return

    if not IS_HOSTED_COLAB:
        print("HF_TOKEN not set in the environment; continuing with anonymous HF access.")
        return

    try:
        from google.colab import userdata
    except ImportError:
        print("Colab secrets are unavailable; continuing without HF_TOKEN.")
        return

    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        os.environ["HF_TOKEN"] = token
        print("Loaded HF_TOKEN from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets; continuing with anonymous HF access.")


def resolve_local_repo_dir() -> Path:
    candidates: list[Path] = []
    env_repo_dir = os.environ.get("KEELNET_REPO_DIR")
    if env_repo_dir:
        candidates.append(Path(env_repo_dir).expanduser())
    candidates.append(DEFAULT_LOCAL_REPO_DIR)
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    seen: set[Path] = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / ".git").exists() and (resolved / "pyproject.toml").exists():
            return resolved

    raise FileNotFoundError(
        "Could not find the KeelNet repo. Set KEELNET_REPO_DIR to your local checkout before running this cell."
    )


def ensure_repo() -> Path:
    if not IS_HOSTED_COLAB:
        return resolve_local_repo_dir()

    local_repo_dir = Path(os.environ.get("KEELNET_REPO_DIR", str(DEFAULT_LOCAL_REPO_DIR)))
    if (local_repo_dir / ".git").exists():
        run_setup(["git", "fetch", "origin"], cwd=local_repo_dir)
    else:
        run_setup(["git", "clone", GIT_REPO_URL, str(local_repo_dir)])

    run_setup(["git", "checkout", GIT_BRANCH], cwd=local_repo_dir)
    run_setup(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=local_repo_dir)
    return local_repo_dir.resolve()


PROJECT_STORAGE_DIR = configure_project_storage()
DRIVE_PROJECT_DIR = PROJECT_STORAGE_DIR
configure_hf_token()
REPO_DIR = ensure_repo().resolve()
os.chdir(REPO_DIR)
print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Runtime repo dir: {REPO_DIR}")
CURRENT_BRANCH = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    check=True,
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Git branch: {CURRENT_BRANCH}")
run_setup([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])


<h2 style="color: #1d4ed8;">2. Configure The Run</h2>

Set `AUTHOR_NAME` to your name. This notebook builds the stage-specific `RUN_NAME` automatically as `yourname-stage4-v1`, `yourname-stage4-v2`, `yourname-stage4-v3`, and so on based on completed runs.

Review `TARGET_METRICS`, `SUGGESTED_MODULES`, `SMOKE_TEST_COMMANDS`, and `STAGE_COMMANDS` before you move into implementation.

This cell also prints the important values you should check before running stage commands.

It creates a small `RUN_STARTED.txt` file in the current run folder immediately, so you can confirm the output path is correct before training or evaluation finishes.


In [ ]:
from pathlib import Path
import json
import re
import subprocess
import sys

import torch

REPO_DIR = Path(REPO_DIR).resolve()
PROJECT_STORAGE_DIR = Path(PROJECT_STORAGE_DIR).resolve()
DRIVE_PROJECT_DIR = PROJECT_STORAGE_DIR

# Change only this for each teammate. The notebook builds the stage name and next version automatically.
AUTHOR_NAME = "naufal"
RUN_BASENAME = f"{AUTHOR_NAME}-stage4"
ARTIFACTS_ROOT = PROJECT_STORAGE_DIR / "artifacts" / "stage4_colab"
COMPLETION_MARKER_NAME = "RUN_COMPLETED.txt"

STAGE_TITLE = "Stage 4: Unsupported-Confidence Control"
STAGE_OBJECTIVE = "Use calibrated QA and support signals to reduce confident unsupported answers without collapsing usefulness."
TARGET_METRICS = [
    "unsupported-answer rate",
    "supported-answer rate",
    "answer F1",
    "abstain F1",
    "answer rate",
]
IMPLEMENTATION_HINTS = [
    "input: Stage 3 calibrated QA and support confidence signals",
    "output: a fixed constrained controller with explicit support and QA gates",
    "beat the Stage 2 gated baseline under an unsupported-answer budget",
]
SUGGESTED_MODULES = ["keelnet.control", "keelnet.calibration", "keelnet.metrics"]


def completed_versions(root: Path, run_basename: str) -> list[int]:
    versions: list[int] = []
    if not root.exists():
        return versions

    pattern = re.compile(rf"^{re.escape(run_basename)}-v(\d+)$")
    for child in root.iterdir():
        if not child.is_dir():
            continue
        match = pattern.match(child.name)
        if match and (child / COMPLETION_MARKER_NAME).exists():
            versions.append(int(match.group(1)))
    return sorted(versions)


RUN_VERSION = max(completed_versions(ARTIFACTS_ROOT, RUN_BASENAME), default=0) + 1
RUN_NAME = f"{RUN_BASENAME}-v{RUN_VERSION}"
OUTPUT_ROOT = ARTIFACTS_ROOT / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_MARKER_FILE = OUTPUT_ROOT / "RUN_STARTED.txt"
RUN_NOTES_FILE = OUTPUT_ROOT / "run-notes-template.md"
RUN_SUMMARY_FILE = OUTPUT_ROOT / "run-summary.json"
COMPLETION_MARKER_FILE = OUTPUT_ROOT / COMPLETION_MARKER_NAME

DEFAULT_STAGE3_CALIBRATION_EVAL = Path("/content/KeelNet-local/artifacts/stage3_colab/naufal-stage3-v1/calibration_eval.json")
CALIBRATION_EVAL_PATH = str(DEFAULT_STAGE3_CALIBRATION_EVAL) if DEFAULT_STAGE3_CALIBRATION_EVAL.exists() else None
EVAL_BATCH_SIZE = 16
MAX_EVAL_SAMPLES = None
MAX_UNSUPPORTED_ANSWER_RATE = 20.0
SUPPORT_THRESHOLD_MIN = 0.40
SUPPORT_THRESHOLD_MAX = 0.80
SUPPORT_THRESHOLD_STEP = 0.05
QA_THRESHOLD_MIN = 0.40
QA_THRESHOLD_MAX = 0.80
QA_THRESHOLD_STEP = 0.05
JOINT_THRESHOLD_MIN = 0.45
JOINT_THRESHOLD_MAX = 0.85
JOINT_THRESHOLD_STEP = 0.05
ALPHA_MIN = 0.50
ALPHA_MAX = 0.90
ALPHA_STEP = 0.10

RUN_SMOKE_TEST = False
SMOKE_TEST_EVAL_SAMPLES = 64

if CALIBRATION_EVAL_PATH is not None:
    CALIBRATION_EVAL_PATH = Path(CALIBRATION_EVAL_PATH).expanduser().resolve()
    if not CALIBRATION_EVAL_PATH.exists():
        raise FileNotFoundError(f"Calibration eval file not found: {CALIBRATION_EVAL_PATH}")

CONTROL_EVAL = OUTPUT_ROOT / "control_eval.json"


def maybe_add_arg(cmd: list[str], flag: str, value: object | None) -> None:
    if value is None:
        return
    cmd.extend([flag, str(value)])


def build_control_command(
    output_path: Path,
    *,
    max_eval_samples: int | None,
    support_threshold_step: float,
    qa_threshold_step: float,
    joint_threshold_step: float,
    alpha_step: float,
) -> list[str] | None:
    if CALIBRATION_EVAL_PATH is None:
        return None

    cmd = [
        sys.executable,
        "-m",
        "keelnet.control",
        "--calibration-path",
        str(CALIBRATION_EVAL_PATH),
        "--output-path",
        str(output_path),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
        "--max-unsupported-answer-rate",
        str(MAX_UNSUPPORTED_ANSWER_RATE),
        "--support-threshold-min",
        str(SUPPORT_THRESHOLD_MIN),
        "--support-threshold-max",
        str(SUPPORT_THRESHOLD_MAX),
        "--support-threshold-step",
        str(support_threshold_step),
        "--qa-threshold-min",
        str(QA_THRESHOLD_MIN),
        "--qa-threshold-max",
        str(QA_THRESHOLD_MAX),
        "--qa-threshold-step",
        str(qa_threshold_step),
        "--joint-threshold-min",
        str(JOINT_THRESHOLD_MIN),
        "--joint-threshold-max",
        str(JOINT_THRESHOLD_MAX),
        "--joint-threshold-step",
        str(joint_threshold_step),
        "--alpha-min",
        str(ALPHA_MIN),
        "--alpha-max",
        str(ALPHA_MAX),
        "--alpha-step",
        str(alpha_step),
    ]
    maybe_add_arg(cmd, "--max-eval-samples", max_eval_samples)
    return cmd


smoke_command = build_control_command(
    OUTPUT_ROOT / "smoke-control-eval.json",
    max_eval_samples=SMOKE_TEST_EVAL_SAMPLES,
    support_threshold_step=0.10,
    qa_threshold_step=0.10,
    joint_threshold_step=0.10,
    alpha_step=0.20,
)
stage_command = build_control_command(
    CONTROL_EVAL,
    max_eval_samples=MAX_EVAL_SAMPLES,
    support_threshold_step=SUPPORT_THRESHOLD_STEP,
    qa_threshold_step=QA_THRESHOLD_STEP,
    joint_threshold_step=JOINT_THRESHOLD_STEP,
    alpha_step=ALPHA_STEP,
)

SMOKE_TEST_COMMANDS = [smoke_command] if smoke_command is not None else []
STAGE_COMMANDS = [stage_command] if stage_command is not None else []

RUN_MARKER_FILE.write_text(
    "\n".join(
        [
            f"stage={STAGE_TITLE}",
            f"run_name={RUN_NAME}",
            f"run_version=v{RUN_VERSION}",
            f"runtime_mode={RUNTIME_MODE}",
            f"repo_dir={REPO_DIR}",
            f"project_storage_dir={PROJECT_STORAGE_DIR}",
            f"git_branch={CURRENT_BRANCH}",
            f"calibration_eval_path={CALIBRATION_EVAL_PATH}",
            "status=configured",
            "note=This file is created when the config cell runs.",
        ]
    )
    + "\n",
    encoding="utf-8",
)

print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Repo dir: {REPO_DIR}")
print(f"{PROJECT_STORAGE_LABEL}: {PROJECT_STORAGE_DIR}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Run basename: {RUN_BASENAME}")
print(f"Run version: v{RUN_VERSION}")
print(f"Run output dir: {OUTPUT_ROOT}")
print(f"Run marker file: {RUN_MARKER_FILE}")
print(f"Calibration eval path: {CALIBRATION_EVAL_PATH}")
print(f"Control eval file: {CONTROL_EVAL}")
print(f"Max unsupported-answer rate: {MAX_UNSUPPORTED_ANSWER_RATE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Target metrics:", ", ".join(TARGET_METRICS))
print("Suggested modules:", ", ".join(SUGGESTED_MODULES))

if CALIBRATION_EVAL_PATH is None:
    print("Set CALIBRATION_EVAL_PATH before running Stage 4.")


def run(cmd):
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    with subprocess.Popen(
        rendered,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    ) as process:
        if process.stdout is not None:
            for line in process.stdout:
                print(line, end="", flush=True)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, rendered)


def run_many(commands, *, label: str) -> bool:
    if not commands:
        print(f"No commands configured for {label} yet.")
        return False

    for index, cmd in enumerate(commands, start=1):
        print(f"\n[{label} {index}/{len(commands)}]")
        run(cmd)
    return True


<h2 style="color: #1d4ed8;">3. Validate The Environment</h2>

Run the project tests before stage-specific work. This confirms the installed code path is at least minimally healthy inside the current runtime.


In [ ]:
run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "tests")])
                


<h2 style="color: #1d4ed8;">4. Optional Smoke Test</h2>

Use this cell only after you fill in `SMOKE_TEST_COMMANDS` in the config cell. Keep those commands tiny so you can catch path, dependency, and runtime issues before a full Stage 4 run.


In [ ]:
if RUN_SMOKE_TEST:
    ran_smoke = run_many(SMOKE_TEST_COMMANDS, label="smoke test")
    if not ran_smoke:
        print("Add one or more small commands to SMOKE_TEST_COMMANDS in the config cell.")
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True after you define SMOKE_TEST_COMMANDS.")
                


<div style="border-left: 6px solid #c2410c; background: #fff7ed; padding: 12px 16px; border-radius: 8px;">
<strong>Implementation Starts Here</strong><br/>
Sections 1-4 are setup and validation. Section 5 onward is the main Stage 4 work area.
<ul>
  <li><strong>Start here:</strong> create <code>src/keelnet/control.py</code> or an equivalent control module, then combine calibrated answer, support, and confidence signals in evaluation.</li>
  <li><strong>Finish here:</strong> confident unsupported answers go down and the result clearly beats the permissive Stage 2 gate.</li>
  <li><strong>Out of scope:</strong> retrieval, adaptive balancing, unrelated model redesigns.</li>
</ul>
</div>


## IMPLEMENTATION 1: Run The Stage 4 Control Commands

This is the main Stage 4 implementation and run section. Everything before this point is setup, sync, or validation.

Fill in `STAGE_COMMANDS` in the config cell with the actual commands for this stage. Start with one command, make sure the outputs land in `OUTPUT_ROOT`, then add the rest.


In [ ]:
if CALIBRATION_EVAL_PATH is None:
    print("Set CALIBRATION_EVAL_PATH in the config cell before running Stage 4.")
else:
    run_many(STAGE_COMMANDS, label="stage command")


## Stage Note Template

Keep your stage notes inside this notebook flow. Update them at three points:

1. before implementation: fill in the goal, success condition, and planned commands
2. after smoke test: record environment issues and command fixes
3. after a meaningful run: record metrics, verdict, and next actions

Use this structure for the generated run note:

- Run info
- Goal
- Commands
- Main metrics
- What changed
- What worked
- What failed or looks risky
- Error cases to review
- Decision
- Next actions


<h2 style="color: #15803d;">Save Notes And Review Artifacts</h2>

This cell creates teammate-friendly note files inside the current run folder and lists the current artifacts. Update the generated notes as you learn what does and does not work in Stage 4: Unsupported-Confidence Control.


In [ ]:
if not RUN_NOTES_FILE.exists():
    metric_lines = [f"- {metric}" for metric in TARGET_METRICS]
    RUN_NOTES_FILE.write_text(
        "\n".join(
            [
                f"# {STAGE_TITLE} Notes",
                "",
                "Update this note three times:",
                "1. before implementation: goal, success condition, and commands",
                "2. after smoke test: environment issues and command fixes",
                "3. after a meaningful run: metrics, verdict, and next actions",
                "",
                "## Run Info",
                f"- Branch: {CURRENT_BRANCH}",
                f"- `RUN_NAME`: {RUN_NAME}",
                f"- Output folder: {OUTPUT_ROOT}",
                f"- Runtime mode: {RUNTIME_MODE}",
                "",
                "## Goal",
                "- One-sentence objective:",
                "- Unsupported-answer budget:",
                "- Out of scope:",
                "",
                "## Commands",
                f"- Smoke test command(s): {SMOKE_TEST_COMMANDS}",
                f"- Main command(s): {STAGE_COMMANDS}",
                f"- Input artifacts or checkpoints: Stage 3 calibration eval at {CALIBRATION_EVAL_PATH}",
                f"- Output files to inspect: {CONTROL_EVAL}",
                "",
                "## Main Metrics",
                *metric_lines,
                "",
                "## What Changed",
                "- ",
                "",
                "## What Worked",
                "- ",
                "",
                "## What Failed Or Looks Risky",
                "- ",
                "",
                "## Error Cases To Review",
                "- ",
                "",
                "## Decision",
                "- Keep, revise, or stop:",
                "- Reason:",
                "",
                "## Next Actions",
                "1. ",
                "2. ",
                "3. ",
            ]
        )
        + "\n",
        encoding="utf-8",
    )

RUN_SUMMARY_FILE.write_text(
    json.dumps(
        {
            "stage": STAGE_TITLE,
            "run_name": RUN_NAME,
            "runtime_mode": RUNTIME_MODE,
            "git_branch": CURRENT_BRANCH,
            "output_root": str(OUTPUT_ROOT),
            "calibration_eval_path": str(CALIBRATION_EVAL_PATH) if CALIBRATION_EVAL_PATH is not None else None,
            "control_eval": str(CONTROL_EVAL),
            "target_metrics": TARGET_METRICS,
            "suggested_modules": SUGGESTED_MODULES,
            "max_unsupported_answer_rate": MAX_UNSUPPORTED_ANSWER_RATE,
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Notes template: {RUN_NOTES_FILE}")
print(f"Run summary: {RUN_SUMMARY_FILE}")
print("Current files under OUTPUT_ROOT:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    print(path)


<h2 style="color: #15803d;">Final Check</h2>

A Stage 4 win is not just fewer unsupported answers.

Check all three:

- unsupported confident answers actually go down
- supported-answer rate and answer quality remain useful
- the behavior change comes from the intended control mechanism, not just over-abstention or another tiny delta

After that, record a few failure cases so Stage 5 can compare this controller against a direct support-constrained learner under matched conditions.


<h2 style="color: #15803d;">Share This Run</h2>

This cell prints a minimal share-ready summary for teammates, saves it into the current run folder, and marks the run as completed so the next run becomes the next version.

Update the metric lines after you review the outputs for this stage.


In [ ]:
from datetime import datetime, timezone

metric_lines = [f"- {metric}: <fill in after review>" for metric in TARGET_METRICS]
if CONTROL_EVAL.exists():
    control_results = json.loads(CONTROL_EVAL.read_text(encoding="utf-8"))
    stage2_metrics = control_results["stage2_gated_dev_metrics"]
    stage4_metrics = control_results["control_dev_metrics"]
    stage2_mix = control_results["stage2_gated_dev_mix"]
    stage4_mix = control_results["control_dev_mix"]
    config = control_results["selected_config"]
    metric_lines = [
        f"- Unsupported-answer rate (dev): {stage2_metrics['unsupported_answer_rate']:.2f} -> {stage4_metrics['unsupported_answer_rate']:.2f}",
        f"- Answerable F1 (dev): {stage2_metrics['answerable_f1']:.2f} -> {stage4_metrics['answerable_f1']:.2f}",
        f"- Overall F1 (dev): {stage2_metrics['overall_f1']:.2f} -> {stage4_metrics['overall_f1']:.2f}",
        f"- Abstain F1 (dev): {stage2_metrics['abstain_f1']:.2f} -> {stage4_metrics['abstain_f1']:.2f}",
        f"- Supported-answer rate (among answers, dev): {stage2_mix['supported_answer_rate']:.2f} -> {stage4_mix['supported_answer_rate']:.2f}",
        f"- Answer rate (dev): {stage2_mix['answer_rate']:.2f} -> {stage4_mix['answer_rate']:.2f}",
        f"- Selected support threshold: {config['support_threshold']:.2f}",
        f"- Selected QA threshold: {config['qa_threshold']:.2f}",
        f"- Selected joint threshold: {config['joint_threshold']:.2f}",
        f"- Selected alpha: {config['alpha']:.2f}",
        f"- Max unsupported-answer rate budget: {control_results['max_unsupported_answer_rate']:.2f}",
    ]

share_lines = [
    f"# {STAGE_TITLE} Share Note",
    "",
    f"- runtime mode: {RUNTIME_MODE}",
    f"- branch name: {CURRENT_BRANCH}",
    f"- RUN_NAME: {RUN_NAME}",
    *metric_lines,
    f"- Output folder path: {OUTPUT_ROOT}",
]
share_note = "\n".join(share_lines) + "\n"
SHARE_NOTE_FILE = OUTPUT_ROOT / "collab-share-note.md"
SHARE_NOTE_FILE.write_text(share_note, encoding="utf-8")
COMPLETION_MARKER_FILE.write_text(
    "\n".join(
        [
            f"run_name={RUN_NAME}",
            f"completed_at={datetime.now(timezone.utc).isoformat()}",
            f"share_note={SHARE_NOTE_FILE.name}",
            "status=completed",
        ]
    )
    + "\n",
    encoding="utf-8",
)
print(share_note)
print(f"Saved share note: {SHARE_NOTE_FILE}")
print(f"Saved completion marker: {COMPLETION_MARKER_FILE}")
